In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
from pathlib import Path

base_path = Path.cwd()
coco_path = base_path / "data" / "coco" 
coco_annotations_path = coco_path / "annotations" 
coco_keypoints_path = coco_annotations_path / "person_keypoints_train2017.json"
coco_scalenet_results_path = coco_path / "coco_results" 
coco_images_root_path =  coco_path / "train2017"

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017
debug = True
coco_scale_train = COCOScale2017(
    debug=debug,
    camera_parameters_file_path=coco_scalenet_results_path / "yannick_results_train2017_filtered",
    coco_json_file_path=coco_keypoints_path,
    coco_image_root_path=coco_images_root_path,
    coco_scale_pickle_path=coco_scalenet_results_path / "results_with_kps_20200208_morethan2_2-8" / "pickle"
)

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_dataset_name = "COCOScale2017_train"
DatasetCatalog.register(coco_scale_dataset_name, coco_scale_train)
MetadataCatalog.get(coco_scale_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

if debug:
    max_vis = 5
    for i, d in enumerate(coco_scale_train):
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5, metadata=MetadataCatalog.get(coco_scale_dataset_name))
        visualizer.draw_dataset_dict(d)
        out = visualizer.get_output()
        img = out.get_image()
        plt.imshow(img)
        plt.show()
        if max_vis == i:
            break

In [ ]:
import os
from detectron2.engine import COCOScaleTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128,expandable_segments:True"

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (coco_scale_dataset_name, )
cfg.DATASETS.TEST = (coco_scale_dataset_name, )
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 16  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.02  # pick a good LR
cfg.MODEL.KEYPOINT_ON = True
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = True  # Althought must be set to false in joined dataset
cfg.DATALOADER.ASPECT_RATIO_GROUPING = True  # Althought must be set to false in joined dataset
cfg.SOLVER.MAX_ITER = 500
experiment_name = "test-debug-kpsbbox-dataset"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.
cfg.FLOAT32_PRECISION = "medium"

trainer = COCOScaleTrainer(cfg) 
# NOTE: change value of resume if we have a last_checkpoint
trainer.resume_or_load(resume=False)
model = trainer.model
# trainer.train()